In [1]:
import sys
!{sys.executable} -m pip install scikit-learn
import sys
!{sys.executable} -m pip install streamlit
import sys
!{sys.executable} -m pip install holidays


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

In [24]:
inferencia_path = "../data/raw/inferencia/ventas_2025_inferencia.csv"

inferencia_df = pd.read_csv(inferencia_path)

inferencia_df.head()


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [25]:
#Duplicados
duplicados = inferencia_df.duplicated().sum()
print(f"El número de filas duplicadas en ventas_df es: {duplicados}")

#Estadistica descriptiva
print(inferencia_df.describe(include='all'))

#Resumen final
if inferencia_df.isnull().sum().sum() ==0:
    print("No hay valores nulos en ventas_df.")
else:
    print("Hay valores nulos en ventas_df.")

if duplicados == 0:
    print("No hay filas duplicadas en ventas_df.")
else:
    print("Hay filas duplicadas en ventas_df.")

El número de filas duplicadas en ventas_df es: 0
             fecha producto_id                    nombre categoria  \
count          888         888                       888       888   
unique          37          24                        24         4   
top     2025-10-25    PROD_001  Nike Air Zoom Pegasus 40   Running   
freq            24          37                        37       296   
mean           NaN         NaN                       NaN       NaN   
std            NaN         NaN                       NaN       NaN   
min            NaN         NaN                       NaN       NaN   
25%            NaN         NaN                       NaN       NaN   
50%            NaN         NaN                       NaN       NaN   
75%            NaN         NaN                       NaN       NaN   
max            NaN         NaN                       NaN       NaN   

              subcategoria  precio_base es_estrella  unidades_vendidas  \
count                  888   888.000

In [26]:
#COnvertimos la columna fecha a tipo datetime en ambos dataframes

inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])


In [27]:
#Variables temporales
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['es_fin_semana']= inferencia_df['dia_semana'].isin(['Saturday', 'Sunday'])


#Festivos España
festivos_es = holidays.CountryHoliday('ES', years=inferencia_df['año'].unique())
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)

#Black Friday (ultimo viernes de noviembre)

def es_black_friday(fecha):
    if fecha.month == 11:
        nov = pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30', freq='D')
        fridays = nov[nov.weekday == 4]
        return fecha in fridays[-1:] 
    return False
inferencia_df['es_black_friday'] = inferencia_df['fecha'].apply(es_black_friday)

#Cyber monday (lunes siguiente al Black Friday)
def es_cyber_monday(fecha):
    if fecha.month == 11 or fecha.month == 12:
        nov = pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30', freq='D')
        fridays = nov[nov.weekday == 4]
        black_friday = fridays[-1]
        cyber_monday = black_friday + pd.Timedelta(days=3)
        return fecha == cyber_monday
    return False
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].apply(es_cyber_monday)

#variable: trimestr

trimestre_map ={1:1,2:1,3:1,4:2,5:2,6:2,7:3,8:3,9:3,10:4,11:4,12:4}
inferencia_df['trimestre'] = inferencia_df['mes'].map(trimestre_map)

#variable semana del anio
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week

#variable: dia laborable (ni fin de semana ni festivo)
inferencia_df['es_laborable'] = inferencia_df['es_fin_semana'] & inferencia_df['es_festivo']

#variable inicio /fin de mes
inferencia_df['es_inicio_mes'] = inferencia_df['fecha'].dt.day <= 3
inferencia_df['es_fin_mes'] = inferencia_df['dia_mes'] >= (inferencia_df['fecha'] + pd.offsets.MonthEnd(0)).dt.day - 2
#Mostramos
print(inferencia_df[['fecha','año','mes','dia_mes','dia_semana','es_fin_semana','es_festivo','es_black_friday','es_cyber_monday']].head())


/var/folders/r0/96mlrgd13m30sgcx6bnhr9yh0000gn/T/ipykernel_1626/2874314012.py:11: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos_es)


       fecha   año  mes  dia_mes dia_semana  es_fin_semana  es_festivo  \
0 2025-10-25  2025   10       25   Saturday           True       False   
1 2025-10-25  2025   10       25   Saturday           True       False   
2 2025-10-25  2025   10       25   Saturday           True       False   
3 2025-10-25  2025   10       25   Saturday           True       False   
4 2025-10-25  2025   10       25   Saturday           True       False   

   es_black_friday  es_cyber_monday  
0            False            False  
1            False            False  
2            False            False  
3            False            False  
4            False            False  


In [30]:
#Cargamos df procesado para obtener los ultimos valores historicos por producto
df_historico = pd.read_csv("../data/procesed/df.csv")
df_historico['fecha'] = pd.to_datetime(df_historico['fecha'])
#lags y media mobil de 7 dias
lags = range(1, 8)
#ordenamos por fecha
inferencia_df = inferencia_df.sort_values(['producto_id', 'fecha'])

for producto_id in inferencia_df['producto_id'].unique():
    #obtener ultimos 7 registros del historico para este producto
    df_prod_hist = df_historico[df_historico['producto_id'] == producto_id].sort_values('fecha').tail(7)
    valores_hist = df_prod_hist['unidades_vendidas'].values

    #Obterner registros de inferencia para este producto
    mask_prod = inferencia_df['producto_id'] == producto_id
    indices_prod = inferencia_df[mask_prod].index

    #inicializar lags con valores historicos
    for i, idx in enumerate(indices_prod):
        for lag in lags:
            if i >= lag:
                #usamos valores dentro de inferencia
                inferencia_df.loc[idx, f'unidades_vendidas_lag{lag}'] = inferencia_df.loc[indices_prod[i-lag], 'unidades_vendidas']
            else:
                #usamos valores del historico
                hist_idx = len(valores_hist) - lag + i
                if hist_idx >= 0:
                    inferencia_df.loc[idx, f'unidades_vendidas_lag{lag}'] = valores_hist[hist_idx]
                else:
                    inferencia_df.loc[idx, f'unidades_vendidas_lag{lag}'] = np.nan
        
        #Calculamos media movil de 7 dias
    if i >= 6:
        #usamos valores de inferencia_df
        valores_ma = [inferencia_df.loc[indices_prod[j], 'unidades_vendidas'] for j in range(i-6, i+1)]
    else:
        valores_ma = []
        for j in range(7):
            if i-j>= 0:
                valores_ma.insert(0, inferencia_df.loc[indices_prod[i-j], 'unidades_vendidas'])
            else:
                hist_idx = len(valores_hist) + i-j
                if hist_idx >= 0:
                    valores_ma.insert(0, valores_hist[hist_idx])
            if len(valores_ma) == 7:
                inferencia_df.loc[idx, 'unidades_vendidas_ma7'] = np.mean(valores_ma)
            else:
                inferencia_df.loc[idx, 'unidades_vendidas_ma7'] = np.nan

print(f'Registros despues de crear lags: {len(inferencia_df)}')

print('Registros con lags')



Registros despues de crear lags: 888
Registros con lags


In [31]:
#variables descuento porcentaje
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta']-inferencia_df['precio_base'])/inferencia_df['precio_base'])*100

#mostramos
print(inferencia_df[['fecha','producto_id','precio_base','precio_venta','descuento_porcentaje']].head())

        fecha producto_id  precio_base  precio_venta  descuento_porcentaje
0  2025-10-25    PROD_001          115        113.13             -1.626087
24 2025-10-26    PROD_001          115        105.75             -8.043478
48 2025-10-27    PROD_001          115        114.95             -0.043478
72 2025-10-28    PROD_001          115        117.31              2.008696
96 2025-10-29    PROD_001          115        108.10             -6.000000


In [32]:
if 'Amazon' in inferencia_df.columns and 'Decathlon' in inferencia_df.columns and 'Deporvillage' in inferencia_df.columns:
    #variable precio competencia: promedio Amazon, decathlon y deporvillage
    inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)

    #variable ratio
    inferencia_df['ratio_precio']= inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

    #Eliminar las variables amazon, dechatlon y deporvillage
    inferencia_df = inferencia_df.drop(columns=['Amazon', 'Decathlon', 'Deporvillage'])
else:
    for producto_id in inferencia_df['producto_id'].unique():
        precio_comp_hist = df_historico[df_historico['producto_id']==producto_id]['precio_competencia'].mean()
        inferencia_df.loc[inferencia_df['producto_id']==producto_id, 'precio_competencia'] = precio_comp_hist

    inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

print(inferencia_df[['fecha','producto_id','precio_venta','precio_competencia','ratio_precio']].head())

        fecha producto_id  precio_venta  precio_competencia  ratio_precio
0  2025-10-25    PROD_001        113.13          102.573333      1.102918
24 2025-10-26    PROD_001        105.75           98.356667      1.075169
48 2025-10-27    PROD_001        114.95           97.740000      1.176079
72 2025-10-28    PROD_001        117.31          103.146667      1.137313
96 2025-10-29    PROD_001        108.10          100.520000      1.075408


In [33]:
#creamos copia de las variables con sufijo _h para one-hot_encoding
inferencia_df['nombre_h']= inferencia_df['nombre']
inferencia_df['categoria_h']= inferencia_df['categoria']
inferencia_df['subcategoria_h']= inferencia_df['subcategoria']

#one hot encoding
inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], drop_first=False)

#aseguramos de que inferencia_df tenga las mismas columnas que el df 
columnas_df = df_historico.columns.tolist()

#aòadir columnas faltantees con valor false
for col in columnas_df:
    if col not in inferencia_df.columns:
        if col.startswith('nombre_h') or col.startswith('categoria_h') or col.startswith('subcategoria_h'):
            inferencia_df[col] = False
        elif col not in ['fecha', 'producto_id', 'nombre','categoria','subcategoria','precio_base','es_estrella','unidades_vendidas','precio_venta','ingresos']:
            inferencia_df[col] = 0
#reordenar columnas para que coincidan con df_historico
inferencia_df = inferencia_df[columnas_df]
print(f'Columnas en df_historico: {len(columnas_df)}')
print(f'Columnas en inferencia_df despues de ajuste: {len(inferencia_df.columns)}')

Columnas en df_historico: 79
Columnas en inferencia_df despues de ajuste: 79


In [35]:
#filtrado y guardado final

#filtrado y guardado final
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].copy()

print(f'Registros después del filtro noviembre: {len(inferencia_df)}')

cols_lag_ma = [f'unidades_vendidas_lag{lag}' for lag in range(1,8)] + ['unidades_vendidas_ma7']

#verificar solo columnas existentes
cols_lag_ma = [col for col in cols_lag_ma if col in inferencia_df.columns]

nulos_noviembre = inferencia_df[cols_lag_ma].isnull().sum()

if nulos_noviembre.sum() > 0:
    print(f'Hay {nulos_noviembre.sum()} registros con nulos en variables de lag o media móvil en noviembre.')
    print(nulos_noviembre[nulos_noviembre > 0])
else:
    print('No hay registros con nulos en variables de lag o media móvil en noviembre.')

Registros después del filtro noviembre: 720
No hay registros con nulos en variables de lag o media móvil en noviembre.


In [36]:

#Guardar el Dataframe
inferencia_df.to_csv("../data/procesed/inferencia_df.csv", index=False)
print("Preprocesamiento completado y archivo guardado en ../data/procesed/inferencia_df.csv")

Preprocesamiento completado y archivo guardado en ../data/procesed/inferencia_df.csv


In [37]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'año', 'mes', 'dia_semana', 'es_fin_semana', 'es_festivo',
       'es_black_friday', 'es_cyber_monday', 'trimestre', 'semana_año',
       'es_laborable', 'es_inicio_mes', 'dia_mes', 'es_fin_mes',
       'unidades_vendidas_lag_1', 'unidades_vendidas_lag_2',
       'unidades_vendidas_lag_3', 'unidades_vendidas_lag_4',
       'unidades_vendidas_lag_5', 'unidades_vendidas_lag_6',
       'unidades_vendidas_lag_7', 'unidades_vendidas_ma_7',
       'porcentaje_descuento', 'precio_competenci', 'precio_competencia',
       'ratio_precio', 'nombre_h_Adidas Own The Run Jacket',
       'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25',
       'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900',
       'nombre_h_Domyos Kit Mancuernas 2